# SFT

## Entry Point

### Supervised Fine-Tuning on Single Node

```
FORCE_TORCHRUN=1 llamafactory-cli train examples/train_full/llama3_full_sft.yaml
```

* `FORCE_TORCHRUN=1`: Forces the use of `torchrun` (PyTorch's native distributed launcher)
* `llamafactory-cli` is the main command-line interface for LLaMA-Factory framework.
    * Entry Point Definition:
        ```python
        # setup.py line 36
        # Main file: src/llamafactory/cli.py
        # Entry function: main() function
        console_scripts = ["llamafactory-cli = llamafactory.cli:main"]
        ```
    * The CLI is a command dispatcher that routes to different modules:
        ```python
        COMMAND_MAP = {
            "api": run_api,
            "chat": run_chat,
            "env": print_env,
            "eval": run_eval,
            "export": export_model,
            "train": run_exp,  # from .train.tuner import export_model, run_exp
            "webchat": run_web_demo,
            "webui": run_web_ui,
            "version": partial(print, WELCOME),
            "help": partial(print, USAGE),
        }
        ```
    * When you install LLaMA-Factory (`pip install -e .`), the `setup.py` registers `llamafactory-cli` as a console script, making it globally available in your environment.

### llama3_full_sft.yaml

```yaml
### model
model_name_or_path: meta-llama/Meta-Llama-3-8B-Instruct
trust_remote_code: true  # a security flag in Hugging Face Transformers that controls whether to allow execution of custom code downloaded from remote model repositories.

### method
stage: sft
do_train: true
finetuning_type: full
deepspeed: examples/deepspeed/ds_z3_config.json  # choices: [ds_z0_config.json, ds_z2_config.json, ds_z3_config.json]

### dataset
dataset: identity,alpaca_en_demo
template: llama3
cutoff_len: 2048  # maximum sequence length for the entire input sequence (input + output combined)
max_samples: 1000
overwrite_cache: true  # slower startup, but guarantees fresh data processing
preprocessing_num_workers: 16
dataloader_num_workers: 4

### output
output_dir: saves/llama3-8b/full/sft
logging_steps: 10
save_steps: 500
plot_loss: true
overwrite_output_dir: true
save_only_model: false
report_to: none  # choices: [none, wandb, tensorboard, swanlab, mlflow]

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 2
learning_rate: 1.0e-5
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 180000000
resume_from_checkpoint: null

### eval
# eval_dataset: alpaca_en_demo
# val_size: 0.1
# per_device_eval_batch_size: 1
# eval_strategy: steps
# eval_steps: 500
```

## Preparation

## Trainer